This notebook illustrate how the SAR Sentinel-1 IW and Level-2 SWOT KaRin measurement are co-registered spatially in 2 situations:
 1) when the KaRin Swath is kept integrally
 2) when the Karin Swath is truncated to a portion that is close to nadir (in order to compare to a third reference : the Nadir beam from SWOT, independent from KaRin sensor)

In [ ]:
import os
import xarray as xr
from scipy import spatial

In [ ]:
from s1swotcolocs.utils import get_conf_content
import s1swotcolocs
potential_config_file_1 = os.path.join(os.path.dirname(s1swotcolocs.__file__), "localconfig.yml")
potential_config_file_2 = os.path.join(os.path.dirname(s1swotcolocs.__file__), "config.yml")
if not os.path.exists(potential_config_file_1):
    confpath = potential_config_file_2
else:
    confpath = potential_config_file_1
print(confpath)
conf = get_conf_content(confpath)

assets_dir = os.path.join(os.path.dirname(s1swotcolocs.__file__),"assets")
conf

In [ ]:
file_swot_l2 = os.path.join(assets_dir,'SWOT_L2_LR_SSH_WindWave_018_555_20240729T172147_20240729T181315_PIC0_01.nc')
assert os.path.exists(file_swot_l2)
l2wav_s1_iw = os.path.join(assets_dir,'l2-s1a-iw3-wav-dv-20240729t172529-20240729t172557-054978-06b28f-e12.nc')
assert os.path.exists(l2wav_s1_iw)
# files = glob.glob(os.path.join(assets_dir,'seastate_coloc_S*nc'))

# files

## How to select points in SWOT swath

In [ ]:
# if False:
    # file_swot_l2 = '/home/datawork-WW3/PROJECT/SWOT/WindWave/SWOT_L2_LR_SSH_WindWave_003_514_20230919T065618_20230919T074746_PGC0_01.nc'
    # coloc_file = 'seastate_coloc_S1A_IW_SLC__1SDV_20230919T065253_20230919T065323_050393_061162_07E0-iw1_SWOT_L2_WindWave_20230919T065625_PGC0.nc'
    # safe_s1_iw = 'SENTINEL1_DS:S1A_IW_SLC__1SDV_20230919T065253_20230919T065323_050393_061'
    # l2wav_s1_iw = '/home/datawork-cersat-public/cache/project/sarwave/data/products/experiments/slc/iw/l2/2023/262/S1A_IW_WAV__2SDV_20230919T065253_20230919T065323_050393_061162_07E0_E12.SAFE/l2-s1a-iw2-wav-dv-20230919t065254-20230919t065322-050393-061162-e12.nc'
subswath_sar = os.path.basename(l2wav_s1_iw).split('-')[2]
subswath_sar

In [ ]:
dsswot = xr.open_dataset(file_swot_l2)
dsswot["longitude"] = dsswot["longitude"].where(
    dsswot["longitude"] < 180, dsswot["longitude"] - 360
)
dsswot

In [ ]:
ds_iw_l2 = xr.open_dataset(l2wav_s1_iw,engine='h5netcdf',group='intraburst')
ds_iw_l2

In [ ]:
#dsswot['swh_karin'].plot()
from matplotlib import pyplot as plt
plt.scatter(dsswot['longitude'].values.ravel(),dsswot['latitude'].values.ravel(),c=dsswot['swh_karin'].values.ravel())
plt.plot(ds_iw_l2['longitude'].values.ravel(),ds_iw_l2['latitude'].values.ravel(),'r.')

In [ ]:
# subswot = dsswot.isel({'num_lines':slice(1000,1200)})
subswot = dsswot.isel({'num_lines':slice(2800,2990)})
subswot

In [ ]:
import s1swotcolocs
import numpy as np
import s1swotcolocs.seastate_colocs_s1_swot
from importlib import reload
reload(s1swotcolocs.seastate_colocs_s1_swot)
treeswot = s1swotcolocs.seastate_colocs_s1_swot.get_swot_tree(subswot)
treeswot

In [ ]:
onetile_ds = ds_iw_l2.isel({'tile_line':8,'tile_sample':4})
lontile = onetile_ds['longitude'].values
lattile = onetile_ds['latitude'].values
onetile_ds
lontile,lattile

In [ ]:
plt.figure(figsize=(8,6),dpi=110)
plt.title(os.path.basename(l2wav_s1_iw))
plt.plot(ds_iw_l2['longitude'].values.ravel(),ds_iw_l2['latitude'].values.ravel(),'r.',label='SAR IW tiles center',ms=3)
im=plt.scatter(subswot['longitude'].values.ravel(),subswot['latitude'].values.ravel(),c=subswot['swh_karin'].values.ravel())
cb = plt.colorbar()
plt.legend()
plt.grid(True)
cb.set_label('SWH KaRin [m]')

In [ ]:
radius_coloc = 0.05

print(lontile)
neighbors = treeswot.query_ball_point([lontile, lattile], r=radius_coloc)
print(neighbors)
dstclosest,closest_index = treeswot.query([lontile, lattile], k=1)
lineclosest,pixelclosets = np.unravel_index(closest_index,subswot['longitude'].shape)
print('closest_index',closest_index,(lineclosest,pixelclosets))
swot_closest_ds = subswot.isel(num_lines=lineclosest, num_pixels=pixelclosets)
indices = []
condensated_swot = xr.Dataset()
for oneneighbor in neighbors:
    index_original_shape_swot_num_lines, index_original_shape_swot_num_pixels = np.unravel_index(oneneighbor,
                                                                                              subswot['longitude'].shape)
    idxtmp = (int(index_original_shape_swot_num_lines), int(index_original_shape_swot_num_pixels))
    # print(idxtmp)
    indices.append(idxtmp)
# print(indices)
subset = [subswot.isel(num_lines=i, num_pixels=j) for i, j in indices]
len(subset)

In [ ]:
indices

In [ ]:
import geopandas as gpd
import cartopy
import cartopy.crs as ccrs
import cartopy.feature as cfeature
from matplotlib import pyplot as plt
from shapely import wkt
from shapely.geometry import Point
fig = plt.figure(dpi=200,figsize=(18,6))
ax = fig.add_subplot(1, 1, 1, projection=ccrs.PlateCarree()) #false_easting=100,false_northing=100000)




petitswot = dsswot.isel({'num_lines':slice(0,-1,1)})
plt.plot(petitswot['longitude'].values.ravel(),petitswot['latitude'].values.ravel(),'g.',label='SWOT',ms=0.6)
petitsar = plt.plot(ds_iw_l2['longitude'].values.ravel(),ds_iw_l2['latitude'].values.ravel(),'rs',label='SAR '+subswath_sar,ms=2)
plt.plot(*wkt.loads(ds_iw_l2.attrs['footprint']).exterior.xy)
# plt.ylim(60,80)
# plt.xlim(10,25)
delta_bound_lon = 0.4*2 #deg
delta_bound_lat = 0.2*2
plt.ylim(lattile-delta_bound_lat,lattile+delta_bound_lat)
plt.xlim(lontile-delta_bound_lon,lontile+delta_bound_lon)

# plt.title('SAR index : %s'%indexes_sar)
ax.add_feature(cfeature.LAND)
ax.add_feature(cfeature.COASTLINE)
gl = ax.gridlines(draw_labels=True, dms=True, x_inline=False, y_inline=False)
gl.right_labels = False  # Disable labels on the right
gl.top_labels = False    # Optional: disable labels on the top too

for ux,uu in enumerate(subset):
    if ux==0:
        plt.plot(uu['longitude'],uu['latitude'],'bo',ms=1.8,alpha=0.7,label='SWOT selected : %i pts'%len(subset))
    else:
        plt.plot(uu['longitude'],uu['latitude'],'bo',ms=1.8,alpha=0.4)
plt.plot(lontile,lattile,'r+',label='SAR tile center',ms=10)

# add the circle of coloc
circle = Point(lontile, lattile).buffer(radius_coloc)

# Wrap into GeoDataFrame for plotting
gdf = gpd.GeoDataFrame(geometry=[circle], crs=ccrs.PlateCarree()) # "EPSG:4326"

# add closest SWOT (for understanding)
ax.plot(swot_closest_ds['longitude'],swot_closest_ds['latitude'],'m*',label='closest SWOT',alpha=0.5,ms=7)

# Plot
ax = gdf.plot(facecolor='none', edgecolor='green',ax=ax,label='%f° coloc radius, centered on SAR tile')
plt.legend()

# same illustration for points close to nadir 

In [ ]:
import s1swotcolocs
import numpy as np
import s1swotcolocs.seastate_colocs_s1_swot
from importlib import reload
reload(s1swotcolocs.seastate_colocs_s1_swot)
from  s1swotcolocs.seastate_colocs_s1_swot import lines_to_keep_in_swot_swath 
print(lines_to_keep_in_swot_swath)
dsswotl2_closenadir = dsswot.isel({'num_pixels':lines_to_keep_in_swot_swath}) # selection validated in notebook
dsswotl2_closenadir

In [ ]:
# subswot = dsswotl2_closenadir.isel({'num_lines':slice(1000,1500)})
subswot = dsswotl2_closenadir.isel({'num_lines':slice(2800,2990)})

treeswot = s1swotcolocs.seastate_colocs_s1_swot.get_swot_tree(subswot)
treeswot
subswot

In [ ]:
# l2wav_s1_iw = '/home/datawork-cersat-public/cache/project/sarwave/data/products/experiments/slc/iw/l2/2023/262/S1A_IW_WAV__2SDV_20230919T065253_20230919T065323_050393_061162_07E0_E12.SAFE/l2-s1a-iw1-wav-dv-20230919t065253-20230919t065321-050393-061162-e12.nc'
# subswath_sar = os.path.basename(l2wav_s1_iw).split('-')[2]
# ds_iw_l2 = xr.open_dataset(l2wav_s1_iw,engine='h5netcdf',group='intraburst')

In [ ]:
onetile_ds = ds_iw_l2.isel({'tile_line':2,'tile_sample':10})
lontile = onetile_ds['longitude'].values
lattile = onetile_ds['latitude'].values
onetile_ds
lontile,lattile

In [ ]:
plt.plot(ds_iw_l2['longitude'].values.ravel(),ds_iw_l2['latitude'].values.ravel(),'r.')
plt.scatter(subswot['longitude'].values.ravel(),subswot['latitude'].values.ravel(),c=subswot['swh_karin'].values.ravel())
plt.plot(lontile,lattile,'mo')

In [ ]:
radius_coloc = 0.08

print(lontile)
neighbors = treeswot.query_ball_point([lontile, lattile], r=radius_coloc)
print(neighbors)
dstclosest,closest_index = treeswot.query([lontile, lattile], k=1)
lineclosest,pixelclosets = np.unravel_index(closest_index,subswot['longitude'].shape)
print('closest_index',closest_index,(lineclosest,pixelclosets))
swot_closest_ds = subswot.isel(num_lines=lineclosest, num_pixels=pixelclosets)
indices = []
condensated_swot = xr.Dataset()
for oneneighbor in neighbors:
    index_original_shape_swot_num_lines, index_original_shape_swot_num_pixels = np.unravel_index(oneneighbor,
                                                                                              subswot['longitude'].shape)
    idxtmp = (int(index_original_shape_swot_num_lines), int(index_original_shape_swot_num_pixels))
    # print(idxtmp)
    indices.append(idxtmp)
# print(indices)
subset = [subswot.isel(num_lines=i, num_pixels=j) for i, j in indices]
len(subset)

In [ ]:
import geopandas as gpd
import cartopy
import cartopy.crs as ccrs
import cartopy.feature as cfeature
from matplotlib import pyplot as plt
from shapely import wkt
from shapely.geometry import Point
fig = plt.figure(dpi=200,figsize=(18,6))
ax = fig.add_subplot(1, 1, 1, projection=ccrs.PlateCarree()) #false_easting=100,false_northing=100000)




# petitswot = dsswot.isel({'num_lines':slice(0,-1,1)})
plt.plot(subswot['longitude'].values.ravel(),subswot['latitude'].values.ravel(),'g.',label='SWOT',ms=0.6)
petitsar = plt.plot(ds_iw_l2['longitude'].values.ravel(),ds_iw_l2['latitude'].values.ravel(),'rs',label='SAR '+subswath_sar,ms=2)
plt.plot(*wkt.loads(ds_iw_l2.attrs['footprint']).exterior.xy)
# plt.ylim(60,80)
# plt.xlim(10,25)
delta_bound_lon = 0.4*3 #deg
delta_bound_lat = 0.2*3
plt.ylim(lattile-delta_bound_lat,lattile+delta_bound_lat)
plt.xlim(lontile-delta_bound_lon,lontile+delta_bound_lon)

# plt.title('SAR index : %s'%indexes_sar)
ax.add_feature(cfeature.LAND)
ax.add_feature(cfeature.COASTLINE)
gl = ax.gridlines(draw_labels=True, dms=True, x_inline=False, y_inline=False)
gl.right_labels = False  # Disable labels on the right
gl.top_labels = False    # Optional: disable labels on the top too

for ux,uu in enumerate(subset):
    if ux==0:
        plt.plot(uu['longitude'],uu['latitude'],'bo',ms=1.8,alpha=0.7,label='SWOT selected : %i pts'%len(subset))
    else:
        plt.plot(uu['longitude'],uu['latitude'],'bo',ms=1.8,alpha=0.4)
plt.plot(lontile,lattile,'r+',label='SAR tile center',ms=10)

# add the circle of coloc
circle = Point(lontile, lattile).buffer(radius_coloc)

# Wrap into GeoDataFrame for plotting
gdf = gpd.GeoDataFrame(geometry=[circle], crs=ccrs.PlateCarree()) # "EPSG:4326"

# add closest SWOT (for understanding)
ax.plot(swot_closest_ds['longitude'],swot_closest_ds['latitude'],'m*',label='closest SWOT',alpha=0.5,ms=7)

# Plot
ax = gdf.plot(facecolor='none', edgecolor='green',ax=ax,label='%f° coloc radius, centered on SAR tile')
plt.legend()